In [ ]:
# Фрагмент коду виконує підключення необхідних для роботи бібліотек та створення словників
# для вирішення проблеми з різною індексацією областей з використанням бібліотеки urllib.request 
# для пакетного завантаження даних індексу VHI з сервера NOAA.


import pandas as pd
import os
import urllib.request
from datetime import datetime

# словник для ідентифікації областей NOAA (англ алфавіт)
noaa_provinces = {
    1: "Cherkasy", 2: "Chernihiv", 3: "Chernivtsi", 4: "Crimea", 5: "Dnipropetrovsk",
    6: "Donetsk", 7: "Ivano-Frankivsk", 8: "Kharkiv", 9: "Kherson", 10: "Khmelnytskyy",
    11: "Kyiv", 12: "Kyiv City", 13: "Kirovohrad", 14: "Luhansk", 15: "Lviv",
    16: "Mykolayiv", 17: "Odessa", 18: "Poltava", 19: "Rivne", 20: "Sevastopol",
    21: "Sumy", 22: "Ternopil", 23: "Transcarpathia", 24: "Vinnytsya", 25: "Volyn",
    26: "Zaporizhzhya", 27: "Zhytomyr"
}

# нові індекси за українським алфавітом 
ua_index_map = {
    "Vinnytsya": 1, "Volyn": 2, "Dnipropetrovsk": 3, "Donetsk": 4, "Zhytomyr": 5,
    "Transcarpathia": 6, "Zaporizhzhya": 7, "Ivano-Frankivsk": 8, "Kyiv": 9,
    "Kirovohrad": 10, "Luhansk": 11, "Lviv": 12, "Mykolayiv": 13, "Odessa": 14,
    "Poltava": 15, "Rivne": 16, "Sumy": 17, "Ternopil": 18, "Kharkiv": 19,
    "Kherson": 20, "Khmelnytskyy": 21, "Cherkasy": 22, "Chernivtsi": 23,
    "Chernihiv": 24, "Crimea": 25
}

In [ ]:
# Функція download_vhi_data забезпечує автоматизоване пакетне завантаження файлів із 
# VHI-індексами для 27 адміністративних одиниць України з сервера NOAA. Створює цільову 
# директорію vhi_data та в циклі генерує URL-запити для кожної області. Для оптимізації 
# роботи та запобігання колізіям реалізовано механізм перевірки наявності файлів у директорії: 
# якщо дані для конкретної області вже існують, повторне довантаження скасовується. Завантажені 
# дані зберігаються локально, причому до назви кожного нового файлу динамічно додається унікальна 
# мітка з поточною датою та часом timestamp, а блок try-except гарантує стабільну роботу 
# скрипту у разі виникнення мережевих помилок.

def download_vhi_data():
    if not os.path.exists('vhi_data'):
        os.makedirs('vhi_data')
        
    for province_id in range(1, 28):
        # формую URL
        url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
        
        # перевірка чи є файл для цієї області
        existing = [f for f in os.listdir('vhi_data') if f.startswith(f"vhi_id_{province_id}_")]
        if existing:
            print(f"Область {province_id} вже завантажена: {existing[0]}")
            continue
            
        try:
            with urllib.request.urlopen(url) as response:
                content = response.read().decode('utf-8')
                # додаю дату і час завантаження до імені
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                filename = f"vhi_id_{province_id}_{timestamp}.csv"
                filepath = os.path.join('vhi_data', filename)
                
                with open(filepath, 'w') as f:
                    f.write(content)
                print(f"Збережено: {filename}")
        except Exception as e:
            print(f"Помилка при завантаженні області {province_id}: {e}")

# виклик
download_vhi_data()

Збережено: vhi_id_1_20260310_162745.csv
Збережено: vhi_id_2_20260310_162746.csv
Збережено: vhi_id_3_20260310_162748.csv
Збережено: vhi_id_4_20260310_162749.csv
Збережено: vhi_id_5_20260310_162750.csv
Збережено: vhi_id_6_20260310_162752.csv
Збережено: vhi_id_7_20260310_162753.csv
Збережено: vhi_id_8_20260310_162754.csv
Збережено: vhi_id_9_20260310_162756.csv
Збережено: vhi_id_10_20260310_162757.csv
Збережено: vhi_id_11_20260310_162758.csv
Збережено: vhi_id_12_20260310_162759.csv
Збережено: vhi_id_13_20260310_162800.csv
Збережено: vhi_id_14_20260310_162801.csv
Збережено: vhi_id_15_20260310_162802.csv
Збережено: vhi_id_16_20260310_162803.csv
Збережено: vhi_id_17_20260310_162805.csv
Збережено: vhi_id_18_20260310_162806.csv
Збережено: vhi_id_19_20260310_162807.csv
Збережено: vhi_id_20_20260310_162808.csv
Збережено: vhi_id_21_20260310_162809.csv
Збережено: vhi_id_22_20260310_162810.csv
Збережено: vhi_id_23_20260310_162812.csv
Збережено: vhi_id_24_20260310_162813.csv
Збережено: vhi_id_25_2026

In [ ]:
# Функція load_and_clean_data виконує зчитування, очищення та консолідацію завантажених 
# текстових файлів у єдиний об'єкт pandas DataFrame. Під час ітерації по директорії скрипт 
# витягує оригінальний ідентифікатор з назви файлу та зчитує дані, пропускаючи службові 
# заголовки NOAA. Процедура очищення включає видалення залишкових HTML-тегів у стовпці з 
# роками, відкидання останнього рядка зі сміттєвими символами та вилучення записів із 
# пропущеними значеннями (dropna). Після цього виконується приведення ключових полів до 
# числових типів (int, float) та заміна ідентифікаторів областей на нові індекси за українською 
# абеткою за допомогою попередньо створених словників. На завершення всі оброблені таблиці 
# об'єднуються (pd.concat) у загальний структурований набір даних, повністю готовий для подальшого аналізу.

def load_and_clean_data(folder='vhi_data'):
    all_data = []
    files = os.listdir(folder)
    
    for filename in files:
        # витягую оригінальний NOAA ID з назви файлу
        noaa_id = int(filename.split('_')[2])
        filepath = os.path.join(folder, filename)
        
        # читаю файл, пропускаючи заголовки NOAA 
        df = pd.read_csv(filepath, index_col=False, header=1, names=['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI'])
        
        df = df.drop(df.index[-1]) # остання строка часто містить </html>
        df['Year'] = df['Year'].str.replace('<tt><pre>', '').str.replace('</pre></tt>', '')
        df = df.dropna()
        
        # в числа
        df = df.astype({'Year': int, 'Week': int, 'VHI': float})
        
        # міняю індекс області на український
        province_name = noaa_provinces[noaa_id]
        if province_name in ua_index_map:
            df['area'] = ua_index_map[province_name]
            all_data.append(df)
            
    result_df = pd.concat(all_data, ignore_index=True)
    return result_df

df = load_and_clean_data()
df.head()

Дані успішно очищено та об'єднано!


,Year,Week,SMN,SMT,VCI,TCI,VHI,area
0,1982,1,0.059,258.24,51.11,48.78,49.95,21
1,1982,2,0.063,261.53,55.89,38.20,47.04,21
2,1982,3,0.063,263.45,57.30,32.69,44.99,21
3,1982,4,0.061,265.10,53.96,28.62,41.29,21
4,1982,5,0.058,266.42,46.87,28.57,37.72,21


In [6]:
# Блок реалізує процедури формування цільових вибірок та статистичного аналізу.
# Додано функції для пошуку за одним роком, за діапазоном років, а також 
# розширений пошук екстремумів (включно із середнім та медіаною).

def get_vhi_by_year(df, area_id, year):
    # ряд VHI для області за вказаний рік
    return df[(df['area'] == area_id) & (df['Year'] == year)][['Week', 'VHI']]

def get_vhi_range(df, areas_list, year_start, year_end):
    # ряд VHI за діапазон років для вказаних областей
    return df[(df['area'].isin(areas_list)) & 
              (df['Year'] >= year_start) & 
              (df['Year'] <= year_end)]

def get_vhi_stats(df, area_id, year):
    # пошук мін, макс, середнього та медіани для області
    subset = df[(df['area'] == area_id) & (df['Year'] == year)]
    return {
        'max': subset['VHI'].max(),
        'min': subset['VHI'].min(),
        'mean': subset['VHI'].mean(),
        'median': subset['VHI'].median()
    }

print("1. Приклад вибірки (Область 1 за 2020 рік):")
print(get_vhi_by_year(df, 1, 2020).head())

print("\n2. Приклад діапазону (Області 1 та 2 за 2000-2002 роки):")
print(get_vhi_range(df, [1, 2], 2000, 2002).head())

print("\n3. Статистика (Область 1 за 2020 рік):")
print(get_vhi_stats(df, 1, 2020))

1. Приклад вибірки (Область 1 за 2020 рік):
       Week    VHI
31044     1  40.92
31045     2  43.19
31046     3  44.74
31047     4  45.29
31048     5  44.80

2. Приклад діапазону (Області 1 та 2 за 2000-2002 роки):
       Year  Week    SMN     SMT    VCI    TCI    VHI  area
30004  2000     1  0.023  260.25   8.46  39.97  24.22     1
30005  2000     2  0.023  259.38  10.27  45.13  27.70     1
30006  2000     3  0.024  259.61  14.76  46.60  30.68     1
30007  2000     4  0.027  260.21  19.52  45.57  32.55     1
30008  2000     5  0.030  260.59  22.24  47.21  34.73     1

3. Статистика (Область 1 за 2020 рік):
{'max': np.float64(64.12), 'min': np.float64(34.48), 'mean': np.float64(45.911538461538456), 'median': np.float64(44.230000000000004)}
